In [3]:
import pandas as pd
import gdown

FILE_ID = "10LGWcTxkq-1xe6kctOZP-WtBpD77_FGp"
gdown.download(id=FILE_ID, output="dados.csv", quiet=False)

tabela = pd.read_csv("dados.csv")
print(tabela.head())
#vou renomear as colunas TotalCharges e Churn
tabela = tabela.rename(columns={'TotalCharges': 'GastoTotal', 'Churn': 'Cancelamentos', 'PaymentMethod': 'Forma_Pagamento', 'InternetService': 'Servico_Internet', 'TechSupport': 'Suporte_Tecnico', 'OnlineSecurity': 'Seguranca_Online', 'OnlineBackup': 'Backup_Online', 'DeviceProtection': 'Protecao_Dispositivo', 'StreamingTV': 'Streaming_TV', 'StreamingMovies': 'Streaming_Filmes', 'MonthlyCharges': 'Cobranca_Mensal', 'tenure': 'Meses_Contrato', 'Contract': 'Tipo_Contrato'})
#na coluna de GastoTotal tinha espaços de texto e foi convertido para numero com a linha de codigo abaixo.
tabela['GastoTotal'] = pd.to_numeric(tabela['GastoTotal'], errors ='coerce')
print(tabela.isnull().sum())
#excluindo as linhas que estavam sem nada da coluna GastoTotal
tabela = tabela.dropna(subset=['GastoTotal'])

#vou tratar a coluna Cancelamentos agora e vou ver quantos clientes Cancelaram
print(tabela['Cancelamentos'].value_counts()) #tivemos 1869 cancelamentos
print((tabela['Cancelamentos'].value_counts(normalize = True)* 100).round(1)) #multiplicando por 100 para ver a porcentagem de cancelamentos, que foi de 26.5% dos clientes.
#agora vou comparar a porcentagem de cancelamento mensal, anual e bianual.
resultado = tabela.groupby('Tipo_Contrato')['Cancelamentos'].value_counts(normalize=True) * 100 #porcentagem de cancelamento por tipo de contrato (mensal, anual e bianual) com os YES e NO
print(resultado.round(1).xs('Yes', level='Cancelamentos')) #isolei para apenas os YES
print(tabela)
colunas_servico = ['Servico_Internet', 'Seguranca_Online', 'Backup_Online',
                    'Protecao_Dispositivo', 'Suporte_Tecnico', 'Streaming_TV',
                    'Streaming_Filmes', 'MultipleLines']



for coluna in ['Servico_Internet', 'Seguranca_Online', 'Suporte_Tecnico',
               'Backup_Online', 'Protecao_Dispositivo']:
    print(coluna)
    cancelados = tabela[tabela['Cancelamentos'] == 'Yes']

    # taxa de cancelamento dentro do grupo (já visto)
    taxa = ((tabela['Cancelamentos'] == 'Yes').groupby(tabela[coluna]).mean() * 100).round(1)

    # % que esse grupo representa do total de cancelamentos
    participacao = (cancelados[coluna].value_counts(normalize=True) * 100).round(1)

    resultado = pd.DataFrame({'Taxa dentro do grupo (%)': taxa,
                              'Participação no total cancelado (%)': participacao})
    print(resultado)
    print()

Downloading...
From: https://drive.google.com/uc?id=10LGWcTxkq-1xe6kctOZP-WtBpD77_FGp
To: c:\Users\gusta\OneDrive\Projeto Dados 2026\dados.csv
100%|██████████| 978k/978k [00:00<00:00, 22.4MB/s]


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [4]:
print(tabela['Seguranca_Online'].value_counts())
print((tabela['Seguranca_Online'].value_counts(normalize=True) * 100).round(1))

Seguranca_Online
No                     3497
Yes                    2015
No internet service    1520
Name: count, dtype: int64
Seguranca_Online
No                     49.7
Yes                    28.7
No internet service    21.6
Name: proportion, dtype: float64


In [5]:
#Esses resultados podem nos levar ao indice que seria basicamente a o grupo de pessoas que nao tem segurança online (49,7%) e o grupo de cancelamentos (78,2). Para chegar ao indice fazemos a seguinte conta:78,2 ÷ 49,7 = 1,57 e o resultado é igual a 57%.

#traduzindo para python: 
indice = 78.2 / 49.7
print(round(indice, 2))

1.57


In [6]:
#vamos usar o seguinte bloco de codigo para responder a pergunta 2
print(pd.crosstab(tabela['Servico_Internet'], tabela['Seguranca_Online']))
print((pd.crosstab(tabela['Servico_Internet'], tabela['Seguranca_Online'], normalize='index') * 100).round(1))

Seguranca_Online    No  No internet service   Yes
Servico_Internet                                 
DSL               1240                    0  1176
Fiber optic       2257                    0   839
No                   0                 1520     0
Seguranca_Online    No  No internet service   Yes
Servico_Internet                                 
DSL               51.3                  0.0  48.7
Fiber optic       72.9                  0.0  27.1
No                 0.0                100.0   0.0


In [7]:
print(tabela)
#fiquei em duvida sobre qual nome tinha colocado em uma coluna e usei isso para consultar de forma rapida. No caso a coluna foi "Meses_Contrato"

      customerID  gender  SeniorCitizen Partner Dependents  Meses_Contrato  \
0     7590-VHVEG  Female              0     Yes         No               1   
1     5575-GNVDE    Male              0      No         No              34   
2     3668-QPYBK    Male              0      No         No               2   
3     7795-CFOCW    Male              0      No         No              45   
4     9237-HQITU  Female              0      No         No               2   
...          ...     ...            ...     ...        ...             ...   
7038  6840-RESVB    Male              0     Yes        Yes              24   
7039  2234-XADUH  Female              0     Yes        Yes              72   
7040  4801-JZAZL  Female              0     Yes        Yes              11   
7041  8361-LTMKD    Male              1     Yes         No               4   
7042  3186-AJIEK    Male              0      No         No              66   

     PhoneService     MultipleLines Servico_Internet Seguranca_

In [ ]:
#Aqui iremos fazer uma analise da pergunta 3.
# Filtra a tabela para pegar só os clientes que têm o pior combo:
# Fibra Ótica + sem Segurança Online
pior_grupo = tabela[(tabela['Servico_Internet'] == 'Fiber optic') & (tabela['Seguranca_Online'] == 'No')]

# Dentro do pior grupo (Fibra Ótica + sem Segurança Online),
# calcula a % de cancelamento por tipo de contrato
print(((pior_grupo['Cancelamentos'] == 'Yes').groupby(pior_grupo['Tipo_Contrato']).mean() * 100).round(1))

Tipo_Contrato
Month-to-month    58.1
One year          20.8
Two year          10.2
Name: Cancelamentos, dtype: float64


In [9]:
print(tabela.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Meses_Contrato', 'PhoneService', 'MultipleLines', 'Servico_Internet', 'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivo', 'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes', 'Tipo_Contrato', 'PaperlessBilling', 'Forma_Pagamento', 'Cobranca_Mensal', 'GastoTotal', 'Cancelamentos']


In [10]:
#para finalizar vamos a pergunta 4 para ver se a motivação do cancelamento é o valor da mensalidade, o que claramente nao foi.
# Compara o valor médio da mensalidade (Cobranca_Mensal) entre quem tem
# e quem não tem Segurança Online
print(tabela.groupby('Seguranca_Online')['Cobranca_Mensal'].mean().round(2))

Seguranca_Online
No                     75.69
No internet service    21.08
Yes                    78.87
Name: Cobranca_Mensal, dtype: float64


In [12]:
# Salva a tabela já tratada (colunas renomeadas, GastoTotal numérico,
# sem linhas vazias) num novo arquivo CSV, pronto para o Power BI
tabela.to_csv('dados/telco_churn_tratado.csv', index=False)